# Face Detection Lanjutan: Haar Cascade vs Haar + Preprocessing + Profile vs YuNet (DNN)
**Laporan UAS Praktikum Visi Komputer**

Notebook ini adalah pengembangan dari program UTS. Masalah di UTS: Haar Cascade frontal kurang optimal saat wajah menoleh ke samping, pencahayaan rendah, dan hasil kotak kurang pas.

Tiga metode dibandingkan:
1. **Haar Baseline**: kode UTS (frontal saja, tanpa preprocessing)
2. **Haar Improved**: CLAHE + koreksi gamma + cascade frontal & profil (termasuk profil di-flip) + NMS
3. **YuNet (DNN)**: detektor wajah berbasis deep learning dari OpenCV Zoo

Kontrol saat webcam berjalan: `1`/`2`/`3` ganti metode, `s` simpan snapshot, `q` keluar.

## 1. Import library
OpenCV untuk deteksi, NumPy untuk array, Pandas untuk tabel evaluasi, Matplotlib untuk visualisasi.

In [ ]:
# !pip install opencv-python numpy matplotlib pandas   # jalankan sekali jika belum terpasang
import os, time, urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("OpenCV versi:", cv2.__version__)   # YuNet butuh OpenCV >= 4.8

## 2. Memuat model
- Haar frontal & profil sudah ada di instalasi OpenCV.
- YuNet (`.onnx`, ±230 KB) diunduh otomatis sekali dari repositori resmi OpenCV Zoo ke folder `models/`.

In [ ]:
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)
YUNET_PATH = os.path.join(MODEL_DIR, "face_detection_yunet_2023mar.onnx")
YUNET_URL = ("https://github.com/opencv/opencv_zoo/raw/main/models/"
             "face_detection_yunet/face_detection_yunet_2023mar.onnx")

if not os.path.exists(YUNET_PATH) or os.path.getsize(YUNET_PATH) < 50_000:
    print("Mengunduh model YuNet ...")
    try:
        urllib.request.urlretrieve(YUNET_URL, YUNET_PATH)
    except Exception as e:
        raise RuntimeError(
            "Unduh otomatis gagal. Unduh manual file face_detection_yunet_2023mar.onnx dari "
            "https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet "
            f"lalu taruh di folder '{MODEL_DIR}/'. Detail: {e}")

frontal = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
profile = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_profileface.xml")
if frontal.empty() or profile.empty():
    raise RuntimeError("Model Haar Cascade gagal dimuat.")

yunet = cv2.FaceDetectorYN.create(YUNET_PATH, "", (320, 320),
                                  score_threshold=0.7, nms_threshold=0.3, top_k=5000)
print("Semua model berhasil dimuat.")

## 3. Fungsi preprocessing (mengatasi pencahayaan rendah)
- **Koreksi gamma otomatis**: mencerahkan frame hanya jika rata-rata kecerahan rendah.
- **CLAHE** (Contrast Limited Adaptive Histogram Equalization): meningkatkan kontras lokal sehingga fitur wajah lebih jelas bagi Haar.

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def auto_gamma(bgr, dark_thr=90, target=110):
    """Cerahkan frame bila rata-rata kecerahan < dark_thr."""
    mean = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean()
    if mean >= dark_thr:
        return bgr
    gamma = np.log(target / 255) / np.log(max(mean, 1) / 255)
    gamma = float(np.clip(gamma, 0.4, 1.0))
    lut = np.array([(i / 255) ** gamma * 255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(bgr, lut)

def enhance_gray(bgr):
    """BGR -> gamma -> grayscale -> CLAHE."""
    gray = cv2.cvtColor(auto_gamma(bgr), cv2.COLOR_BGR2GRAY)
    return _clahe.apply(gray)

## 4. Tiga fungsi detektor
Semua mengembalikan list `(x, y, w, h, score)`; `score = None` untuk Haar karena Haar tidak memberi skor kepercayaan.

- **Haar Baseline**: persis kode UTS.
- **Haar Improved**: frontal + profil kiri + profil kanan (dengan flip horizontal), lalu hasil digabung dengan **NMS** agar kotak ganda dibuang.
- **YuNet**: detektor CNN ringan yang menangani wajah miring/menoleh dan cahaya redup jauh lebih baik, serta memberi skor kepercayaan.

In [ ]:
def merge_boxes(boxes, nms_thr=0.3):
    """Gabungkan kotak yang tumpang tindih (Non-Maximum Suppression)."""
    if len(boxes) == 0:
        return []
    b = [[int(x), int(y), int(w), int(h)] for x, y, w, h in boxes]
    scores = [float(w * h) for _, _, w, h in b]          # kotak lebih besar diprioritaskan
    idx = np.array(cv2.dnn.NMSBoxes(b, scores, 0.0, nms_thr)).flatten()
    return [tuple(b[i]) + (None,) for i in idx]

def detect_haar_baseline(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = frontal.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return [(int(x), int(y), int(w), int(h), None) for x, y, w, h in faces]

def detect_haar_improved(frame):
    gray = enhance_gray(frame)
    W = gray.shape[1]
    kw = dict(scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    boxes = list(frontal.detectMultiScale(gray, **kw))
    boxes += list(profile.detectMultiScale(gray, **kw))                 # profil menghadap satu sisi
    for x, y, w, h in profile.detectMultiScale(cv2.flip(gray, 1), **kw):  # sisi sebaliknya
        boxes.append((W - x - w, y, w, h))
    return merge_boxes(boxes)

def detect_yunet(frame):
    h, w = frame.shape[:2]
    yunet.setInputSize((w, h))
    _, faces = yunet.detect(frame)
    if faces is None:
        return []
    return [(int(f[0]), int(f[1]), int(f[2]), int(f[3]), float(f[-1])) for f in faces]

METHODS = {
    "1": ("Haar Baseline",  detect_haar_baseline, (0, 0, 255)),    # merah
    "2": ("Haar Improved",  detect_haar_improved, (255, 128, 0)),  # biru muda
    "3": ("YuNet (DNN)",    detect_yunet,         (0, 200, 0)),    # hijau
}

def draw_detections(img, dets, color, label=None):
    for x, y, w, h, s in dets:
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        if s is not None:
            cv2.putText(img, f"{s:.2f}", (x, max(y - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    if label:
        cv2.putText(img, label, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    return img

# pemanasan agar pengukuran waktu pertama tidak bias
_dummy = np.zeros((480, 640, 3), np.uint8)
for _, fn, _ in METHODS.values():
    fn(_dummy)

## 5. Deteksi real-time lewat webcam
Metode bisa diganti langsung saat berjalan (tombol `1`, `2`, `3`) sehingga perbedaannya bisa dilihat pada wajah yang sama. Teks di layar menampilkan metode aktif, jumlah wajah, FPS, dan waktu deteksi. Frame terakhir dikembalikan untuk ditampilkan di notebook.

In [ ]:
def run_realtime(camera_index=0, start_mode="3", save_dir="hasil"):
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError("Webcam tidak dapat dibuka.")
    mode, last_frame, fps, prev = start_mode, None, 0.0, time.time()
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Frame gagal dibaca.")
                break
            name, fn, color = METHODS[mode]
            t0 = time.perf_counter()
            dets = fn(frame)
            ms = (time.perf_counter() - t0) * 1000

            now = time.time()
            fps = 0.9 * fps + 0.1 * (1 / max(now - prev, 1e-6)); prev = now

            draw_detections(frame, dets, color)
            cv2.putText(frame, f"{name} | Faces: {len(dets)} | {fps:.0f} FPS | {ms:.0f} ms",
                        (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
            cv2.putText(frame, "1/2/3: metode | s: simpan | q: keluar",
                        (10, frame.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            last_frame = frame.copy()
            cv2.imshow("Face Detection UAS", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("s"):
                path = os.path.join(save_dir, f"snap_{int(time.time())}.png")
                cv2.imwrite(path, frame); print("Disimpan:", path)
            elif key != 255 and chr(key) in METHODS:
                mode = chr(key)
    finally:
        cap.release()
        cv2.destroyAllWindows()
    return last_frame

last_frame = run_realtime(start_mode="3")

In [ ]:
if last_frame is not None:
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(last_frame, cv2.COLOR_BGR2RGB))
    plt.title("Frame terakhir hasil deteksi"); plt.axis("off"); plt.show()
else:
    print("Tidak ada frame yang berhasil diproses.")

## 6. Evaluasi terstruktur (untuk Bab Hasil Pengujian)
Setiap skenario diambil satu frame (tekan **SPASI**, **q** untuk melewati), lalu ketiga metode dijalankan pada **frame yang sama** supaya perbandingan adil. `ground_truth` = jumlah wajah sebenarnya di frame.

Metrik: jumlah terdeteksi, apakah tepat (`terdeteksi == ground_truth`), salah positif (kelebihan deteksi), terlewat (wajah tidak terdeteksi), dan waktu (ms). Ini evaluasi berbasis **jumlah wajah**, bukan IoU.

In [ ]:
SCENARIOS = [  # (nama skenario, jumlah wajah sebenarnya)
    ("Frontal, cahaya baik", 1),
    ("Dua wajah menghadap kamera", 2),
    ("Miring ringan (~15 derajat)", 1),
    ("Menoleh ~45 derajat", 1),
    ("Profil samping ~90 derajat", 1),
    ("Pencahayaan rendah", 1),
    ("Backlight (cahaya dari belakang)", 1),
    ("Jarak jauh (~2-3 m)", 1),
    ("Tanpa wajah", 0),
]

def evaluate_frame(frame, scenario, gt):
    rows, panels = [], []
    for key, (name, fn, color) in METHODS.items():
        t0 = time.perf_counter(); dets = fn(frame); ms = (time.perf_counter() - t0) * 1000
        n = len(dets)
        rows.append(dict(skenario=scenario, ground_truth=gt, metode=name, terdeteksi=n,
                         tepat=(n == gt), salah_positif=max(n - gt, 0),
                         terlewat=max(gt - n, 0), waktu_ms=round(ms, 1)))
        panels.append(draw_detections(frame.copy(), dets, color, f"{name}: {n}"))
    return rows, panels

def run_evaluation(camera_index=0, save_dir="hasil"):
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError("Webcam tidak dapat dibuka.")
    all_rows, all_panels = [], {}
    try:
        for scenario, gt in SCENARIOS:
            snap = None
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                view = frame.copy()
                cv2.putText(view, f"{scenario} (GT={gt})", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                cv2.putText(view, "SPASI: ambil | q: lewati", (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
                cv2.imshow("Evaluasi", view)
                k = cv2.waitKey(1) & 0xFF
                if k == 32: snap = frame.copy(); break
                if k == ord("q"): break
            if snap is None:
                continue
            rows, panels = evaluate_frame(snap, scenario, gt)
            all_rows += rows; all_panels[scenario] = panels
            for p, (name, _, _) in zip(panels, METHODS.values()):
                cv2.imwrite(os.path.join(save_dir, f"{scenario[:20].replace(' ', '_')}_{name[:5]}.png"), p)
    finally:
        cap.release(); cv2.destroyAllWindows()
    return pd.DataFrame(all_rows), all_panels

df, panels = run_evaluation()
df

### Visualisasi per skenario (kiri ke kanan: Haar Baseline, Haar Improved, YuNet)

In [ ]:
for scenario, ps in panels.items():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    for ax, p in zip(axes, ps):
        ax.imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB)); ax.axis("off")
    fig.suptitle(scenario); plt.tight_layout(); plt.show()

### Ringkasan hasil per metode dan grafik

In [ ]:
ringkasan = df.groupby("metode").agg(
    akurasi_hitung=("tepat", "mean"),
    total_salah_positif=("salah_positif", "sum"),
    total_terlewat=("terlewat", "sum"),
    rata2_waktu_ms=("waktu_ms", "mean"),
).round(2)
display(ringkasan)

tabel = df.pivot(index="skenario", columns="metode", values="terdeteksi")
tabel.insert(0, "ground_truth", df.drop_duplicates("skenario").set_index("skenario")["ground_truth"])
display(tabel)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ringkasan["akurasi_hitung"].plot.bar(ax=ax[0], color=["#d33", "#e90", "#2a2"], title="Akurasi jumlah wajah")
ringkasan["rata2_waktu_ms"].plot.bar(ax=ax[1], color=["#d33", "#e90", "#2a2"], title="Rata-rata waktu deteksi (ms)")
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

df.to_csv("hasil_evaluasi.csv", index=False)   # lampirkan di laporan bila perlu

## 7. Penjelasan singkat
| Masalah di UTS | Penyebab pada Haar frontal | Solusi di notebook ini |
|---|---|---|
| Wajah menoleh tidak terdeteksi | Model hanya dilatih untuk wajah **depan** | Cascade profil + flip; atau YuNet |
| Pencahayaan rendah gagal | Fitur Haar bergantung selisih intensitas yang menipis saat gelap | Koreksi gamma + CLAHE; YuNet lebih tahan |
| Kotak kurang pas / deteksi salah | `minNeighbors` tetap, tanpa skor kepercayaan | NMS untuk gabung kotak; YuNet memberi skor dan ambang (`score_threshold`) |

**Rekomendasi:** YuNet sebagai metode utama karena akurat, tetap real-time di CPU, dan terintegrasi di OpenCV. Haar Improved dipertahankan sebagai pembanding.

**Catatan:** evaluasi berbasis jumlah wajah dengan sampel kecil, jadi hasilnya indikatif, bukan benchmark umum. Untuk evaluasi lebih ketat gunakan dataset berlabel (mis. WIDER FACE) dengan metrik IoU/precision/recall.